# 제주도 신재생에너지 발전량 예측 - EDA

## 목표
- 기상 데이터와 발전량 데이터의 특성 파악
- 데이터 품질 검증
- 변수 간 상관관계 분석
- 시계열 패턴 파악

In [ ]:
# 라이브러리 임포트
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# 한글 폰트 설정 (Mac)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 시각화 스타일
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# 경고 숨기기
import warnings
warnings.filterwarnings('ignore')

print('✅ 라이브러리 로드 완료')

## 1. 데이터 로드

In [ ]:
# 데이터 경로
DATA_DIR = '../data'

# 통합 데이터 로드
df = pd.read_csv(f'{DATA_DIR}/processed/merged_dataset.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.set_index('datetime')

print(f'📊 데이터 shape: {df.shape}')
print(f'📅 기간: {df.index.min()} ~ {df.index.max()}')
print(f'\n컬럼: {df.columns.tolist()}')

In [ ]:
# 기본 정보
df.info()

In [ ]:
# 통계 요약
df.describe().round(2)

## 2. 데이터 품질 검증

In [ ]:
# 결측치 확인
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    '결측치 수': missing,
    '비율(%)': missing_pct
})

print('📋 결측치 현황')
print(missing_df[missing_df['결측치 수'] > 0])

if missing.sum() == 0:
    print('\n✅ 결측치 없음')

In [ ]:
# 시간 연속성 확인
time_diff = df.index.to_series().diff().dropna()
expected_diff = pd.Timedelta(hours=1)

gaps = time_diff[time_diff != expected_diff]
print(f'📅 시간 간격 검사')
print(f'   예상 간격: {expected_diff}')
print(f'   불연속 구간: {len(gaps)}개')

if len(gaps) > 0:
    print(f'\n⚠️ 불연속 구간 상세:')
    print(gaps.head(10))

In [ ]:
# 이상치 탐지 (IQR 방식)
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower) | (data[column] > upper)]
    return len(outliers), lower, upper

print('📊 이상치 탐지 (IQR)')
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    n_outliers, lower, upper = detect_outliers_iqr(df, col)
    pct = n_outliers / len(df) * 100
    if n_outliers > 0:
        print(f'   {col}: {n_outliers}개 ({pct:.1f}%) - 범위: [{lower:.2f}, {upper:.2f}]')

## 3. 타겟 변수 분석 (발전량)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 태양광 발전량 분포
axes[0, 0].hist(df['solar_mw'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('발전량 (MW)')
axes[0, 0].set_ylabel('빈도')
axes[0, 0].set_title('태양광 발전량 분포')
axes[0, 0].axvline(df['solar_mw'].mean(), color='r', linestyle='--', label=f'평균: {df["solar_mw"].mean():.1f}')
axes[0, 0].legend()

# 풍력 발전량 분포
axes[0, 1].hist(df['wind_mw'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].set_xlabel('발전량 (MW)')
axes[0, 1].set_ylabel('빈도')
axes[0, 1].set_title('풍력 발전량 분포')
axes[0, 1].axvline(df['wind_mw'].mean(), color='r', linestyle='--', label=f'평균: {df["wind_mw"].mean():.1f}')
axes[0, 1].legend()

# 시간대별 평균 발전량
hourly_solar = df.groupby(df.index.hour)['solar_mw'].mean()
hourly_wind = df.groupby(df.index.hour)['wind_mw'].mean()

axes[1, 0].plot(hourly_solar.index, hourly_solar.values, 'o-', label='태양광', linewidth=2)
axes[1, 0].plot(hourly_wind.index, hourly_wind.values, 's-', label='풍력', linewidth=2)
axes[1, 0].set_xlabel('시간')
axes[1, 0].set_ylabel('평균 발전량 (MW)')
axes[1, 0].set_title('시간대별 평균 발전량')
axes[1, 0].legend()
axes[1, 0].set_xticks(range(0, 24, 3))

# 월별 평균 발전량
monthly_solar = df.groupby(df.index.month)['solar_mw'].mean()
monthly_wind = df.groupby(df.index.month)['wind_mw'].mean()

x = np.arange(12)
width = 0.35
axes[1, 1].bar(x - width/2, monthly_solar.values, width, label='태양광')
axes[1, 1].bar(x + width/2, monthly_wind.values, width, label='풍력')
axes[1, 1].set_xlabel('월')
axes[1, 1].set_ylabel('평균 발전량 (MW)')
axes[1, 1].set_title('월별 평균 발전량')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(['1월', '2월', '3월', '4월', '5월', '6월', 
                            '7월', '8월', '9월', '10월', '11월', '12월'])
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('../data/processed/eda_generation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 기상 변수 분석

In [ ]:
# 기상 변수 컬럼
weather_cols = ['temperature', 'humidity', 'wind_speed', 'pressure', 'solar_radiation', 'cloud_cover']
available_weather = [col for col in weather_cols if col in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(available_weather):
    ax = axes[i]
    ax.hist(df[col].dropna(), bins=50, edgecolor='black', alpha=0.7)
    ax.set_xlabel(col)
    ax.set_ylabel('빈도')
    ax.set_title(f'{col} 분포')
    ax.axvline(df[col].mean(), color='r', linestyle='--', alpha=0.7)

# 빈 subplot 숨기기
for i in range(len(available_weather), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.savefig('../data/processed/eda_weather_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 상관관계 분석

In [ ]:
# 상관계수 히트맵
corr_cols = ['solar_mw', 'wind_mw', 'total_mw'] + available_weather
corr_data = df[corr_cols].copy()

correlation = corr_data.corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, mask=mask, annot=True, cmap='RdBu_r', center=0,
            fmt='.2f', square=True, linewidths=0.5)
plt.title('변수 간 상관관계')
plt.tight_layout()
plt.savefig('../data/processed/eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 타겟 변수와의 상관관계 (중요도 순)
print('📊 태양광 발전량과의 상관관계')
solar_corr = correlation['solar_mw'].drop(['solar_mw', 'wind_mw', 'total_mw']).sort_values(key=abs, ascending=False)
print(solar_corr.round(3))

print('\n📊 풍력 발전량과의 상관관계')
wind_corr = correlation['wind_mw'].drop(['solar_mw', 'wind_mw', 'total_mw']).sort_values(key=abs, ascending=False)
print(wind_corr.round(3))

## 6. 시계열 패턴 분석

In [ ]:
# 7일간 시계열 플롯
sample_week = df.loc['2024-07-01':'2024-07-07']

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# 발전량
axes[0].plot(sample_week.index, sample_week['solar_mw'], label='태양광', linewidth=1.5)
axes[0].plot(sample_week.index, sample_week['wind_mw'], label='풍력', linewidth=1.5)
axes[0].set_ylabel('발전량 (MW)')
axes[0].set_title('발전량 시계열 (2024년 7월 첫째 주)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 기온
if 'temperature' in sample_week.columns:
    axes[1].plot(sample_week.index, sample_week['temperature'], color='red', linewidth=1.5)
    axes[1].set_ylabel('기온 (°C)')
    axes[1].set_title('기온')
    axes[1].grid(True, alpha=0.3)

# 일사량
if 'solar_radiation' in sample_week.columns:
    axes[2].fill_between(sample_week.index, sample_week['solar_radiation'], alpha=0.5, color='orange')
    axes[2].set_ylabel('일사량')
    axes[2].set_title('일사량')
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/eda_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 요일별 패턴
df['dayofweek'] = df.index.dayofweek
df['hour'] = df.index.hour

# 요일-시간 히트맵
pivot_solar = df.pivot_table(values='solar_mw', index='hour', columns='dayofweek', aggfunc='mean')
pivot_wind = df.pivot_table(values='wind_mw', index='hour', columns='dayofweek', aggfunc='mean')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(pivot_solar, ax=axes[0], cmap='YlOrRd', annot=False)
axes[0].set_title('태양광: 요일-시간 평균 발전량')
axes[0].set_xlabel('요일 (0=월요일)')
axes[0].set_ylabel('시간')

sns.heatmap(pivot_wind, ax=axes[1], cmap='YlGnBu', annot=False)
axes[1].set_title('풍력: 요일-시간 평균 발전량')
axes[1].set_xlabel('요일 (0=월요일)')
axes[1].set_ylabel('시간')

plt.tight_layout()
plt.savefig('../data/processed/eda_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. 주요 발견 요약

In [ ]:
print('=' * 60)
print('📋 EDA 주요 발견 요약')
print('=' * 60)

print(f'''
1. 데이터 개요
   - 총 레코드: {len(df):,}개
   - 기간: {df.index.min().date()} ~ {df.index.max().date()}
   - 결측치: {df.isnull().sum().sum()}개

2. 태양광 발전
   - 평균 발전량: {df['solar_mw'].mean():.1f} MW
   - 최대 발전량: {df['solar_mw'].max():.1f} MW
   - 피크 시간: {hourly_solar.idxmax()}시
   - 주요 상관변수: {solar_corr.index[0]} (r={solar_corr.iloc[0]:.2f})

3. 풍력 발전
   - 평균 발전량: {df['wind_mw'].mean():.1f} MW
   - 최대 발전량: {df['wind_mw'].max():.1f} MW
   - 주요 상관변수: {wind_corr.index[0]} (r={wind_corr.iloc[0]:.2f})

4. 모델링 권장사항
   - 태양광: 일사량, 시간, 구름량을 주요 피처로 사용
   - 풍력: 풍속, 기압을 주요 피처로 사용
   - 시간적 특성 (hour, dayofweek, month) 포함 필수
''')

In [ ]:
# 정리
df = df.drop(columns=['dayofweek', 'hour'], errors='ignore')
print('✅ EDA 완료')